# Wood Cutting Problem with AMPL in Python
## AMPL Modeling for Book Problem 3.2

This notebook models the frame-cutting problem from book section `3.2` with AMPL from Python using `amplpy`.


## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.

If your environment does not have `amplpy`, install it first:

```python
%pip install amplpy
```

The first call to `ampl_notebook(modules=["highs"], license_uuid="default")` may download the solver module if it is not already available.


In [1]:
from __future__ import annotations

from functools import wraps
from math import isclose
from time import perf_counter


In [2]:
%pip install amplpy
try:
    from amplpy import ampl_notebook
except ImportError as exc:
    raise ImportError(
        "This notebook requires amplpy. Install it with `%pip install amplpy` before running."
    ) from exc


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed

    return wrapper


def create_ampl_instance(solver: str = "highs"):
    ampl = ampl_notebook(modules=[solver], license_uuid="default")
    ampl.option["solver"] = solver
    ampl.option["solver_msg"] = 0
    return ampl


def round_if_close(value: float, digits: int = 4):
    rounded = round(value, digits)
    return int(round(rounded)) if isclose(rounded, round(rounded), abs_tol=1e-9) else rounded


def extract_1d_solution(variable, keys):
    raw = variable.get_values().to_dict()
    values = {}
    for key in keys:
        if key in raw:
            value = raw[key]
        else:
            value = raw[(key,)]
        values[key] = round_if_close(float(value))
    return values


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: /Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Problem: Efficient Cutting Patterns

The model uses the three efficient `300 cm` cutting patterns presented in the book and minimizes total waste plus inventory overproduction costs for `119 cm` and `90 cm` pieces.


In [3]:
PATTERNS = ["pattern_1", "pattern_2", "pattern_3"]
PIECES = ["pieces_119", "pieces_90"]

WASTE = {"pattern_1": 62, "pattern_2": 1, "pattern_3": 30}
INVENTORY_COST = {"pieces_119": 40, "pieces_90": 30}
DEMAND = {"pieces_119": 350, "pieces_90": 350}
PIECES_PER_PATTERN = {
    ("pattern_1", "pieces_119"): 2,
    ("pattern_1", "pieces_90"): 0,
    ("pattern_2", "pieces_119"): 1,
    ("pattern_2", "pieces_90"): 2,
    ("pattern_3", "pieces_119"): 0,
    ("pattern_3", "pieces_90"): 3,
}
EXPECTED = {
    "patterns": {"pattern_1": 87, "pattern_2": 176, "pattern_3": 0},
    "inventory": {"pieces_119": 0, "pieces_90": 2},
    "cost": 5_630,
}


@timed
def solve_wood_cutting_with_ampl(solver: str = "highs"):
    ampl = create_ampl_instance(solver)
    ampl.eval(
        r'''
        set PATTERNS ordered;
        set PIECES ordered;

        param waste {PATTERNS} >= 0;
        param inventory_cost {PIECES} >= 0;
        param demand {PIECES} >= 0;
        param pieces_per_pattern {PATTERNS, PIECES} >= 0;

        var UsePattern {p in PATTERNS} integer >= 0;
        var Inventory {k in PIECES} integer >= 0;

        minimize TotalCost:
            sum {p in PATTERNS} waste[p] * UsePattern[p]
            + sum {k in PIECES} inventory_cost[k] * Inventory[k];

        subject to PieceDemand {k in PIECES}:
            sum {p in PATTERNS} pieces_per_pattern[p, k] * UsePattern[p] >= demand[k];

        subject to InventoryDefinition {k in PIECES}:
            Inventory[k] = sum {p in PATTERNS} pieces_per_pattern[p, k] * UsePattern[p] - demand[k];
        '''
    )
    ampl.set["PATTERNS"] = PATTERNS
    ampl.set["PIECES"] = PIECES
    ampl.param["waste"] = WASTE
    ampl.param["inventory_cost"] = INVENTORY_COST
    ampl.param["demand"] = DEMAND
    ampl.param["pieces_per_pattern"] = PIECES_PER_PATTERN
    ampl.solve()

    return {
        "patterns": extract_1d_solution(ampl.var["UsePattern"], PATTERNS),
        "inventory": extract_1d_solution(ampl.var["Inventory"], PIECES),
        "cost": round_if_close(ampl.obj["TotalCost"].value()),
    }


ampl_result, ampl_time = solve_wood_cutting_with_ampl()
print("=== WOOD CUTTING RESULTS WITH AMPL ===")
print(f"Solution -> {ampl_result}")
print(f"Time     -> {ampl_time:.8f} seconds")

assert ampl_result == EXPECTED
print("AMPL matches the published cutting plan.")


/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: === WOOD CUTTING RESULTS WITH AMPL ===
Solution -> {'patterns': {'pattern_1': 87, 'pattern_2': 176, 'pattern_3': 0}, 'inventory': {'pieces_119': 0, 'pieces_90': 2}, 'cost': 5630}
Time     -> 1.75759612 seconds
AMPL matches the published cutting plan.
